# Electrochemical Data Analysis Notebook

**Comprehensive analysis notebook for single and multi-file electrochemical experiments**

This notebook replicates the functionality of the Explorer Tab UI with full user control over plotting and analysis.

---

## Features:
- **Simple cell selection** from database
- **Automatic data loading** with full analytics pipeline
- **Interactive plotting** with Plotly
- **Complete data access** to all analytics results
- **User-controlled visualization** and range selection

---

## 🔧 Section 1: Setup & Initialization

Import required libraries and initialize the analysis system.

In [ ]:
# Core imports
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

# Analysis system imports
from src_clean.backend.api import BackendAPI
from src_clean.analysis.registry import get_analysis_registry

# Data manipulation and visualization
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)

print("✅ Libraries imported successfully")

In [ ]:
# Initialize analysis system
try:
    api = BackendAPI()
    registry = get_analysis_registry()
    print("✅ Analysis system initialized successfully")
    print(f"📊 Database path: {api.db.db_path}")
except Exception as e:
    print(f"❌ Failed to initialize analysis system: {e}")
    print("💡 Make sure you're running from the project directory")

## 📋 Section 2: Cell Selection

View available cells and select one for analysis.

In [ ]:
# Get all available cells
cells_data = api.get_cells()
cells_df = pd.DataFrame(cells_data)

if not cells_df.empty:
    print(f"📊 Found {len(cells_df)} cells in database:")
    print("\n" + "="*80)
    display(cells_df[['name', 'chemistry', 'capacity_ah', 'created_at']].head(10))
    print("="*80)
    
    # Show cell names for easy copying
    print("\n🎯 Available cell names for selection:")
    for i, cell_name in enumerate(cells_df['name'].head(10), 1):
        print(f"  {i}. {cell_name}")
else:
    print("❌ No cells found in database")
    print("💡 Process some files first using the UI")

In [ ]:
# 🎯 USER SELECTION: Change this cell name to analyze different data
SELECTED_CELL = "AR3753"  # 👈 CHANGE THIS TO YOUR CELL NAME

print(f"🔍 Selected cell: {SELECTED_CELL}")

# Verify cell exists
if SELECTED_CELL in cells_df['name'].values:
    selected_cell_info = cells_df[cells_df['name'] == SELECTED_CELL].iloc[0]
    print("\n📋 Cell Information:")
    print(f"  Name: {selected_cell_info['name']}")
    print(f"  Chemistry: {selected_cell_info.get('chemistry', 'N/A')}")
    print(f"  Capacity: {selected_cell_info.get('capacity_ah', 'N/A')} Ah")
    print(f"  Created: {selected_cell_info['created_at']}")
    
    # Get files for this cell
    cell_files = api.get_cell_files(SELECTED_CELL)
    print(f"\n📁 Files in cell: {len(cell_files)}")
    for file_info in cell_files:
        print(f"  - {file_info['original_filename']} (ID: {file_info['file_id']})")
else:
    print(f"❌ Cell '{SELECTED_CELL}' not found in database")
    print("💡 Check the cell name spelling or select from the list above")

## 📊 Section 3: Data Loading & Analytics Discovery

Load the comprehensive dataset with all analytics and explore available analysis options.

In [ ]:
# Load comprehensive dataset with all analytics
print(f"🔄 Loading comprehensive dataset for cell: {SELECTED_CELL}")
print("   This includes all raw data + analytics results...")

try:
    # This is the same API call that Explorer Tab uses!
    dataset = api.get_research_dataset_for_perspective(cells=[SELECTED_CELL])
    
    # Convert to pandas for easy manipulation
    if hasattr(dataset, 'to_pandas'):
        dataset_df = dataset.to_pandas()
    else:
        dataset_df = dataset
    
    print(f"✅ Dataset loaded successfully!")
    print(f"📊 Shape: {dataset_df.shape[0]} segments × {dataset_df.shape[1]} columns")
    
    # Quick overview
    print(f"⏱️  Time range: {dataset_df['time_s'].min():.1f}s to {dataset_df['time_s'].max():.1f}s")
    print(f"⚡ Voltage range: {dataset_df['potential_v'].min():.3f}V to {dataset_df['potential_v'].max():.3f}V")
    print(f"🔋 Current range: {dataset_df['current_a'].min():.6f}A to {dataset_df['current_a'].max():.6f}A")
    
    # Show techniques present
    techniques = dataset_df['fundamental_technique'].value_counts()
    print(f"\n🧪 Techniques present:")
    for technique, count in techniques.items():
        print(f"  - {technique}: {count} segments")
        
except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    dataset_df = None

In [ ]:
# Discover available analytics
print("🔍 Discovering available analytics...")

try:
    analytics_options = registry.get_analysis_options()
    analytics_df = pd.DataFrame(analytics_options)
    
    print(f"✅ Found {len(analytics_df)} analytics methods:")
    print("\n" + "="*100)
    display(analytics_df[['analysis_id', 'name', 'applicable_techniques', 'output_columns']].head(10))
    print("="*100)
    
except Exception as e:
    print(f"❌ Failed to load analytics options: {e}")
    analytics_df = None

## 🔍 Section 4: Data Exploration

Explore the dataset structure and categorize columns for easy plotting.

In [ ]:
if dataset_df is not None:
    print("📊 Dataset Overview:")
    print("\n" + "="*80)
    dataset_df.info()
    print("="*80)

In [ ]:
# Categorize columns for easy plotting
if dataset_df is not None:
    all_columns = list(dataset_df.columns)
    
    # Column categories
    time_cols = [col for col in all_columns if 'time' in col.lower()]
    voltage_cols = [col for col in all_columns if any(x in col.lower() for x in ['potential', 'voltage'])]
    current_cols = [col for col in all_columns if 'current' in col.lower()]
    capacity_cols = [col for col in all_columns if 'capacity' in col.lower() or 'cap' in col.lower()]
    energy_cols = [col for col in all_columns if 'energy' in col.lower()]
    analytics_cols = [col for col in all_columns if any(x in col for x in ['time_constant', 'r_squared', 'kinetics', 'analytics'])]
    technique_cols = [col for col in all_columns if 'technique' in col.lower()]
    
    print("🗂️  Column Categories:")
    print(f"\n⏱️  Time columns ({len(time_cols)}):")
    for col in time_cols[:10]:  # Show first 10
        print(f"    {col}")
    
    print(f"\n⚡ Voltage columns ({len(voltage_cols)}):")
    for col in voltage_cols[:10]:
        print(f"    {col}")
    
    print(f"\n🔋 Current columns ({len(current_cols)}):")
    for col in current_cols[:10]:
        print(f"    {col}")
    
    print(f"\n🔋 Capacity columns ({len(capacity_cols)}):")
    for col in capacity_cols[:10]:
        print(f"    {col}")
    
    print(f"\n📊 Analytics columns ({len(analytics_cols)}):")
    for col in analytics_cols[:10]:
        print(f"    {col}")
    
    print(f"\n🧪 Technique columns ({len(technique_cols)}):")
    for col in technique_cols:
        print(f"    {col}")

In [ ]:
# Statistical overview
if dataset_df is not None:
    print("📈 Statistical Overview:")
    print("\n" + "="*100)
    
    # Select key numeric columns for statistics
    key_numeric_cols = ['time_s', 'potential_v', 'current_a', 'duration_s']
    key_numeric_cols = [col for col in key_numeric_cols if col in dataset_df.columns]
    
    if key_numeric_cols:
        display(dataset_df[key_numeric_cols].describe())
    
    print("="*100)

## 📈 Section 5: Plotting Templates & User Visualization

Ready-to-use plotting functions and examples for common electrochemical visualizations.

In [ ]:
# Plotting utility functions
def plot_time_series(x_col, y_cols, title="Time Series Plot", height=600):
    """
    Plot time series data with multiple y-axes.
    
    Args:
        x_col (str): X-axis column name (usually time)
        y_cols (list): List of Y-axis column names
        title (str): Plot title
        height (int): Plot height in pixels
    """
    if dataset_df is None:
        print("❌ No data loaded")
        return
    
    if isinstance(y_cols, str):
        y_cols = [y_cols]
    
    # Filter valid columns
    valid_cols = [col for col in y_cols if col in dataset_df.columns]
    if not valid_cols:
        print(f"❌ No valid columns found from: {y_cols}")
        return
    
    fig = px.line(dataset_df, x=x_col, y=valid_cols, title=title, height=height)
    fig.update_layout(
        xaxis_title=x_col,
        yaxis_title="Value",
        hovermode='x unified'
    )
    fig.show()

def plot_scatter(x_col, y_col, color_col=None, title="Scatter Plot", height=600):
    """
    Create scatter plot with optional color coding.
    
    Args:
        x_col (str): X-axis column name
        y_col (str): Y-axis column name
        color_col (str): Optional color column
        title (str): Plot title
        height (int): Plot height in pixels
    """
    if dataset_df is None:
        print("❌ No data loaded")
        return
    
    fig = px.scatter(dataset_df, x=x_col, y=y_col, color=color_col, title=title, height=height)
    fig.update_layout(
        xaxis_title=x_col,
        yaxis_title=y_col
    )
    fig.show()

def plot_by_technique(x_col, y_col, title="Plot by Technique", height=600):
    """
    Plot data colored by fundamental technique.
    
    Args:
        x_col (str): X-axis column name
        y_col (str): Y-axis column name
        title (str): Plot title
        height (int): Plot height in pixels
    """
    if dataset_df is None:
        print("❌ No data loaded")
        return
    
    fig = px.scatter(dataset_df, x=x_col, y=y_col, 
                    color='fundamental_technique', title=title, height=height)
    fig.update_layout(
        xaxis_title=x_col,
        yaxis_title=y_col
    )
    fig.show()

def filter_data(technique=None, time_range=None, voltage_range=None):
    """
    Filter dataset based on technique, time, or voltage ranges.
    
    Args:
        technique (str): Fundamental technique to filter
        time_range (tuple): (min_time, max_time) in seconds
        voltage_range (tuple): (min_voltage, max_voltage) in volts
    
    Returns:
        pd.DataFrame: Filtered dataset
    """
    if dataset_df is None:
        print("❌ No data loaded")
        return None
    
    filtered_df = dataset_df.copy()
    
    if technique:
        filtered_df = filtered_df[filtered_df['fundamental_technique'] == technique]
        print(f"🔍 Filtered to {technique}: {len(filtered_df)} segments")
    
    if time_range:
        min_time, max_time = time_range
        filtered_df = filtered_df[
            (filtered_df['time_s'] >= min_time) & 
            (filtered_df['time_s'] <= max_time)
        ]
        print(f"⏱️  Filtered to time range {min_time}-{max_time}s: {len(filtered_df)} segments")
    
    if voltage_range:
        min_voltage, max_voltage = voltage_range
        filtered_df = filtered_df[
            (filtered_df['potential_v'] >= min_voltage) & 
            (filtered_df['potential_v'] <= max_voltage)
        ]
        print(f"⚡ Filtered to voltage range {min_voltage}-{max_voltage}V: {len(filtered_df)} segments")
    
    return filtered_df

print("✅ Plotting functions defined successfully!")
print("\n🎨 Available functions:")
print("  - plot_time_series(x_col, y_cols, title)")
print("  - plot_scatter(x_col, y_col, color_col, title)")
print("  - plot_by_technique(x_col, y_col, title)")
print("  - filter_data(technique, time_range, voltage_range)")

## 🎨 Section 6: Example Visualizations

Common electrochemical plots and analytics visualizations.

In [ ]:
# Example 1: Basic time series - Voltage and Current vs Time
if dataset_df is not None:
    print("📊 Example 1: Voltage and Current vs Time")
    plot_time_series('time_s', ['potential_v', 'current_a'], 
                    title=f"Voltage & Current vs Time - {SELECTED_CELL}")

In [ ]:
# Example 2: Technique-based analysis
if dataset_df is not None:
    print("📊 Example 2: Voltage vs Time by Technique")
    plot_by_technique('time_s', 'potential_v', 
                     title=f"Voltage vs Time by Technique - {SELECTED_CELL}")

In [ ]:
# Example 3: Analytics visualization - Time Constants
if dataset_df is not None and 'kinetics_analytics_time_constant_s' in dataset_df.columns:
    print("📊 Example 3: Time Constants Analysis")
    plot_scatter('start_potential_v', 'kinetics_analytics_time_constant_s', 
                'fundamental_technique',
                title=f"Time Constants vs Start Voltage - {SELECTED_CELL}")
else:
    print("ℹ️  Time constants data not available in current dataset")

In [ ]:
# Example 4: Capacity analysis
if dataset_df is not None and 'capacity_ah' in dataset_df.columns:
    print("📊 Example 4: Capacity Analysis")
    plot_time_series('time_s', ['capacity_ah'], 
                    title=f"Capacity vs Time - {SELECTED_CELL}")
else:
    print("ℹ️  Capacity data not available in current dataset")

## 🎯 Section 7: User Playground

**Your space for custom analysis and plotting!**

Use the functions defined above or create your own visualizations.

In [ ]:
# 🎨 USER PLAYGROUND - Customize these plots!

# Quick data check
if dataset_df is not None:
    print(f"📊 Dataset shape: {dataset_df.shape}")
    print(f"⏱️  Time range: {dataset_df['time_s'].min():.1f} - {dataset_df['time_s'].max():.1f} seconds")
    print(f"🧪 Available techniques: {dataset_df['fundamental_technique'].unique()}")
    
    # Show some column options for plotting
    numeric_cols = dataset_df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"\n📈 Available numeric columns for plotting ({len(numeric_cols)}):")
    for i, col in enumerate(numeric_cols[:20], 1):  # Show first 20
        print(f"  {i:2d}. {col}")
    if len(numeric_cols) > 20:
        print(f"     ... and {len(numeric_cols) - 20} more")

In [ ]:
# 🎯 Custom Plot 1: Your choice!
# Example: Uncomment and modify these lines

# plot_time_series('time_s', ['potential_v'], "My Custom Voltage Plot")
# plot_scatter('duration_s', 'capacity_ah', 'fundamental_technique', "Duration vs Capacity")
# plot_by_technique('start_potential_v', 'end_potential_v', "Start vs End Voltage")

print("💡 Uncomment and modify the lines above to create your custom plots!")

In [ ]:
# 🎯 Custom Plot 2: Filtered data analysis
# Example: Analyze only REST phases

# rest_data = filter_data(technique='rest')
# if rest_data is not None and not rest_data.empty:
#     fig = px.scatter(rest_data, x='duration_s', y='potential_v', 
#                     title="REST Phase: Duration vs Voltage")
#     fig.show()

print("💡 Uncomment and modify the lines above to analyze filtered data!")

In [ ]:
# 🎯 Custom Plot 3: Advanced multi-plot
# Example: Create subplot with multiple views

# if dataset_df is not None:
#     from plotly.subplots import make_subplots
#     
#     fig = make_subplots(
#         rows=2, cols=2,
#         subplot_titles=('Voltage vs Time', 'Current vs Time', 
#                        'Voltage vs Current', 'Capacity vs Time')
#     )
#     
#     # Add traces
#     fig.add_trace(go.Scatter(x=dataset_df['time_s'], y=dataset_df['potential_v'], 
#                             mode='lines', name='Voltage'), row=1, col=1)
#     fig.add_trace(go.Scatter(x=dataset_df['time_s'], y=dataset_df['current_a'], 
#                             mode='lines', name='Current'), row=1, col=2)
#     fig.add_trace(go.Scatter(x=dataset_df['potential_v'], y=dataset_df['current_a'], 
#                             mode='markers', name='V vs I'), row=2, col=1)
#     if 'capacity_ah' in dataset_df.columns:
#         fig.add_trace(go.Scatter(x=dataset_df['time_s'], y=dataset_df['capacity_ah'], 
#                                 mode='lines', name='Capacity'), row=2, col=2)
#     
#     fig.update_layout(height=800, title_text=f"Multi-Plot Analysis - {SELECTED_CELL}")
#     fig.show()

print("💡 Uncomment the code above to create an advanced multi-plot analysis!")

## 📝 Section 8: Data Export & Summary

Export data and generate analysis summary.

In [ ]:
# Data export options
if dataset_df is not None:
    print("💾 Data Export Options:")
    print("\n" + "="*60)
    
    # Option 1: Export full dataset
    export_path = f"../data_exports/{SELECTED_CELL}_full_dataset.csv"
    print(f"1. Full dataset export:")
    print(f"   dataset_df.to_csv('{export_path}', index=False)")
    
    # Option 2: Export specific columns
    key_cols = ['time_s', 'potential_v', 'current_a', 'fundamental_technique', 'duration_s']
    key_cols = [col for col in key_cols if col in dataset_df.columns]
    export_path_key = f"../data_exports/{SELECTED_CELL}_key_data.csv"
    print(f"\n2. Key columns export:")
    print(f"   dataset_df{key_cols}.to_csv('{export_path_key}', index=False)")
    
    # Option 3: Export analytics only
    if analytics_cols:
        analytics_export_cols = ['id'] + analytics_cols
        analytics_export_cols = [col for col in analytics_export_cols if col in dataset_df.columns]
        export_path_analytics = f"../data_exports/{SELECTED_CELL}_analytics.csv"
        print(f"\n3. Analytics results export:")
        print(f"   dataset_df{analytics_export_cols}.to_csv('{export_path_analytics}', index=False)")
    
    print("\n" + "="*60)
    print("💡 Uncomment and run the desired export commands above")

In [ ]:
# Analysis summary
if dataset_df is not None:
    print(f"📋 Analysis Summary for Cell: {SELECTED_CELL}")
    print("\n" + "="*80)
    
    # Basic statistics
    print(f"📊 Dataset Overview:")
    print(f"   Total segments: {len(dataset_df):,}")
    print(f"   Time span: {dataset_df['time_s'].max() - dataset_df['time_s'].min():.1f} seconds")
    print(f"   Voltage range: {dataset_df['potential_v'].min():.3f} to {dataset_df['potential_v'].max():.3f} V")
    print(f"   Current range: {dataset_df['current_a'].min():.6f} to {dataset_df['current_a'].max():.6f} A")
    
    # Technique breakdown
    print(f"\n🧪 Technique Distribution:")
    technique_counts = dataset_df['fundamental_technique'].value_counts()
    for technique, count in technique_counts.items():
        percentage = (count / len(dataset_df)) * 100
        print(f"   {technique}: {count:,} segments ({percentage:.1f}%)")
    
    # Analytics availability
    if analytics_cols:
        print(f"\n📊 Analytics Results Available:")
        for col in analytics_cols[:10]:  # Show first 10
            non_null_count = dataset_df[col].notna().sum()
            if non_null_count > 0:
                percentage = (non_null_count / len(dataset_df)) * 100
                print(f"   {col}: {non_null_count:,} values ({percentage:.1f}%)")
    
    print("\n" + "="*80)
    print("✅ Analysis complete!")

---

## 🎉 Analysis Complete!

**This notebook provides full access to your electrochemical data with the power of the Explorer Tab UI, but with complete user control over visualization and analysis.**

### Next Steps:
1. **Modify the `SELECTED_CELL`** variable to analyze different cells
2. **Create custom plots** using the provided functions
3. **Filter and analyze** specific techniques or time ranges
4. **Export data** for external analysis
5. **Extend the notebook** with your own analysis functions

### Key Features:
- ✅ **Single API call** loads all data + analytics
- ✅ **No complex setup** - just change the cell name
- ✅ **Interactive plotting** with Plotly
- ✅ **Easy filtering** and data manipulation
- ✅ **Full analytics access** - same as Explorer Tab

---

**Happy analyzing! 🚀**